# Laboratory Exercise: Search and A*
## Using an LLM as an Engineering Assistant
**Course:** Undergraduate Artificial Intelligence  
**Focus:** Search Problem Formulation, A* Implementation, Heuristic Investigation, BFS Comparison, and Empirical Validation  

---

## 1. Learning Objectives
By the end of this laboratory, you will be able to:
1. Formulate a simple navigation problem as a formal search problem;
2. Identify states, actions, transitions, initial states, goals, and costs;
3. Explain the difference between blind (uninformed) and informed search;
4. Describe the mathematical role of a heuristic in A* search ($f(n) = g(n) + h(n)$);
5. Use an LLM to assist in implementing a search algorithm;
6. Test and validate code generated by an LLM against systematic benchmark test cases;
7. Compare the behavior and efficiency of search algorithms using standard performance metrics;
8. Critically reflect on the use of an LLM as an engineering tool.

---

## 2. Background: Formal Search Formulation
In classical AI, a search problem is represented by the 6-tuple:
$$\mathcal{P} = (S, A, T, s_0, G, c)$$
* $S$: Set of states.
* $A$: Set of actions.
* $T: S \times A \to S$: Transition model.
* $s_0 \in S$: Initial state.
* $G \subseteq S$: Goal state(s).
* $c(s, a, s')$: Step cost function.

### Evaluation Strategies
* **BFS (Breadth-First Search):** Uninformed. Expands shallowest unexpanded node ($f(n) = \text{depth}(n)$).
* **DFS (Depth-First Search):** Uninformed. Expands deepest unexpanded node.
* **Greedy Best-First Search:** Informed. Expands node estimated closest to goal ($f(n) = h(n)$).
* **A\* Search:** Informed. Combines exact path cost from start $g(n)$ and estimated cost to goal $h(n)$:
  $$f(n) = g(n) + h(n)$$
  When $h(n)$ is **admissible** ($0 \le h(n) \le h^*(n)$) and **consistent** (satisfies triangle inequality $h(n) \le c(n, a, n') + h(n')$), A* graph search is guaranteed to return an optimal (shortest) path.


---
## 3. Scenario: Warehouse Robot Navigation
An autonomous warehouse robot moves through a grid with obstacles (shelving units).
The robot moves one cell at a time: **Up, Down, Left, Right**, each with step cost $c = 1$.

```text
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
```

* `#`: Obstacle (impassable wall/shelf).
* `.`: Free navigable cell.
* `S`: Start location $(1, 1)$.
* `G`: Goal location $(7, 15)$.


---
## Task 0: Understand the Search Problem
Before writing any code, we formally specify the problem components.

### Formal Specification Table

| Component | Mathematical Definition | Warehouse Robot Specification |
| :--- | :--- | :--- |
| **State Space $S$** | Set of all valid configurations | Coordinate pair $(r, c)$ where $0 \le r < 9$, $0 \le c < 17$, and $\text{grid}[r][c] \neq \text{'\#'}$. |
| **Actions $A$** | Feasible movements | $\mathcal{A} = \{\text{Up}, \text{Down}, \text{Left}, \text{Right}\}$. |
| **Transition $T(s, a)$** | Deterministic state transition | $T((r, c), \text{Up}) = (r-1, c)$; $T((r, c), \text{Down}) = (r+1, c)$; $T((r, c), \text{Left}) = (r, c-1)$; $T((r, c), \text{Right}) = (r, c+1)$. |
| **Initial State $s_0$** | Starting state | $s_0 = (1, 1)$, the coordinate of symbol `'S'`. |
| **Goal $G$** | Target state predicate | $G = \{(7, 15)\}$, the coordinate of symbol `'G'`. |
| **Cost Function $c$** | Step cost | $c(s, a, s') = 1$ for all valid transitions (uniform step cost). |

---

### Questions

**(a) What information is necessary to specify a state?**  
The discrete grid coordinate tuple $(r, c)$ (row index and column index). Orientation, battery level, or velocity are not modeled in this kinematic navigation problem, so 2D position completely specifies the state.

**(b) What makes an action invalid?**  
An action $a$ is invalid from state $(r, c)$ if:
1. It results in coordinates outside the grid boundaries ($r' < 0$, $r' \ge H$, $c' < 0$, or $c' \ge W$); OR
2. The destination cell contains an obstacle: $\text{grid}[r'][c'] == \text{'\#'}$.

**(c) Is this a deterministic search problem?**  
**Yes.** Given any state $(r, c)$ and an action $a$, the resulting successor state $s' = T(s, a)$ is uniquely and definitively determined with probability $1.0$. There is no stochasticity, sensor noise, or environmental interference.

**(d) What would constitute a solution?**  
A solution is a finite sequence of actions $\langle a_1, a_2, \dots, a_k \rangle$ that generates a sequence of valid states $\langle s_0, s_1, \dots, s_k \rangle$ such that $s_i = T(s_{i-1}, a_i)$, each $a_i$ is valid, and $s_k \in G$.


---
## Task 1: Plan the Agent
Before prompting an LLM for code, we establish the modular architectural design:

1. **State Representation:** A Python tuple of integers `(r, c)`.
2. **Warehouse Representation:** A list of strings or 2D list of characters parsed from the ASCII layout.
3. **Valid Action Check:** `0 <= nr < height and 0 <= nc < width and grid[nr][nc] != '#'`.
4. **Goal Recognition:** Exact coordinate equality `current_state == goal_state`.
5. **Frontier Data Structure:** A min-priority queue (using Python's `heapq` module) storing tuples:  
   `(f_score, tie_breaker_counter, state)` where $f(n) = g(n) + h(n)$.
6. **Path Reconstruction:** A predecessor map `came_from: Dict[State, Tuple[State, Action]]` updated whenever a cheaper path to a state is discovered. Backtracking from $G$ to $S$ extracts the exact sequence.
7. **Report Metrics:**
   * Whether a solution was found (`True`/`False`);
   * The reconstructed path sequence;
   * Path length (number of actions);
   * Number of states expanded.


---
## Task 2: Implementation of A* Search
We implement a robust, fully documented Python search engine supporting both A* (with configurable heuristics) and Breadth-First Search (BFS).


In [1]:
"""
Warehouse Robot Navigation Search Framework
Implements A* Search and BFS with multiple heuristic options.
"""
import heapq
import math
from collections import deque
from typing import List, Tuple, Dict, Optional, Callable

State = Tuple[int, int]
Action = str

class WarehouseSearch:
    def __init__(self, map_str: str):
        self.grid = [list(line.strip()) for line in map_str.strip().splitlines() if line.strip()]
        self.height = len(self.grid)
        self.width = len(self.grid[0])
        self.start = self._find_char('S')
        self.goal = self._find_char('G')

        if not self.start or not self.goal:
            raise ValueError("Map must contain 'S' and 'G'.")

    def _find_char(self, char: str) -> Optional[State]:
        for r in range(self.height):
            for c in range(self.width):
                if self.grid[r][c] == char:
                    return (r, c)
        return None

    def get_successors(self, state: State) -> List[Tuple[State, Action]]:
        r, c = state
        moves = [
            ((-1, 0), "Up"),
            ((1, 0), "Down"),
            ((0, -1), "Left"),
            ((0, 1), "Right")
        ]
        successors = []
        for (dr, dc), action in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < self.height and 0 <= nc < self.width and self.grid[nr][nc] != '#':
                successors.append(((nr, nc), action))
        return successors

    def a_star(self, heuristic_fn: Optional[Callable[[State, State], float]] = None) -> Dict:
        """
        Executes A* search with the given heuristic function h(state, goal).
        """
        if heuristic_fn is None:
            # Default to Manhattan distance
            heuristic_fn = lambda s, g: abs(s[0] - g[0]) + abs(s[1] - g[1])

        start, goal = self.start, self.goal
        counter = 0

        # Frontier: min-heap storing (f_score, counter, current_state)
        frontier = []
        h0 = heuristic_fn(start, goal)
        heapq.heappush(frontier, (h0, counter, start))

        # g_score: shortest known distance from start to state
        g_score: Dict[State, float] = {start: 0.0}

        # Predecessor map: state -> (parent_state, action)
        came_from: Dict[State, Tuple[Optional[State], Optional[Action]]] = {start: (None, None)}

        # Closed set to avoid re-expanding states
        closed_set = set()
        states_expanded = 0

        while frontier:
            f, _, current = heapq.heappop(frontier)

            if current in closed_set:
                continue

            closed_set.add(current)
            states_expanded += 1

            # Goal Test
            if current == goal:
                path, actions = self._reconstruct_path(came_from, goal)
                return {
                    "found": True,
                    "path": path,
                    "actions": actions,
                    "cost": g_score[goal],
                    "path_length": len(actions),
                    "states_expanded": states_expanded
                }

            current_g = g_score[current]

            for next_state, action in self.get_successors(current):
                tentative_g = current_g + 1.0  # Unit cost

                if next_state not in g_score or tentative_g < g_score[next_state]:
                    g_score[next_state] = tentative_g
                    h_val = heuristic_fn(next_state, goal)
                    f_val = tentative_g + h_val
                    counter += 1
                    came_from[next_state] = (current, action)
                    heapq.heappush(frontier, (f_val, counter, next_state))

        return {
            "found": False,
            "path": [],
            "actions": [],
            "cost": float('inf'),
            "path_length": 0,
            "states_expanded": states_expanded
        }

    def bfs(self) -> Dict:
        """
        Executes standard Breadth-First Search (BFS).
        """
        start, goal = self.start, self.goal
        frontier = deque([start])
        visited = {start}
        came_from: Dict[State, Tuple[Optional[State], Optional[Action]]] = {start: (None, None)}
        states_expanded = 0

        while frontier:
            current = frontier.popleft()
            states_expanded += 1

            if current == goal:
                path, actions = self._reconstruct_path(came_from, goal)
                return {
                    "found": True,
                    "path": path,
                    "actions": actions,
                    "cost": len(actions),
                    "path_length": len(actions),
                    "states_expanded": states_expanded
                }

            for next_state, action in self.get_successors(current):
                if next_state not in visited:
                    visited.add(next_state)
                    came_from[next_state] = (current, action)
                    frontier.append(next_state)

        return {
            "found": False,
            "path": [],
            "actions": [],
            "cost": float('inf'),
            "path_length": 0,
            "states_expanded": states_expanded
        }

    def _reconstruct_path(self, came_from, goal):
        path = []
        actions = []
        curr = goal
        while curr is not None:
            path.append(curr)
            parent, act = came_from[curr]
            if act is not None:
                actions.append(act)
            curr = parent
        path.reverse()
        actions.reverse()
        return path, actions

    def render_path(self, path: List[State]) -> str:
        display_grid = [row.copy() for row in self.grid]
        for (r, c) in path:
            if (r, c) != self.start and (r, c) != self.goal:
                display_grid[r][c] = '*'
        return "\n".join("".join(row) for row in display_grid)

print("WarehouseSearch framework successfully defined.")


WarehouseSearch framework successfully defined.


---
## Task 3: Test the Generated Program
We conduct the 4 required systematic test cases:
1. **Test 1: Original Warehouse** (9x17 complex layout)
2. **Test 2: Trivial Case** (1-step immediately adjacent goal)
3. **Test 3: No Solution** (Goal completely enclosed by obstacles)
4. **Test 4: Alternative Paths** (Grid with short path vs long detour)


In [1]:
ORIGINAL_WAREHOUSE = """
#################
#S....#.........#
#.###.#.#######.#
#...#.#.......#.#
###.#.#######.#.#
#...#.........#.#
#.###########.#.#
#.............#G#
#################
"""

TRIVIAL_MAP = """
#####
#SG##
#####
"""

NO_SOLUTION_MAP = """
#######
#S....#
###.###
#...#G#
#######
"""

ALTERNATIVE_PATHS_MAP = """
#########
#S.....G#
#.#####.#
#.......#
#########
"""

print("=================== TEST 1: ORIGINAL WAREHOUSE ===================")
search_orig = WarehouseSearch(ORIGINAL_WAREHOUSE)
res1 = search_orig.a_star()
print(f"Path Found: {res1['found']}")
print(f"Path Length: {res1['path_length']}")
print(f"States Expanded: {res1['states_expanded']}")
print("Solution Path Coordinates:")
print(res1['path'])
print("ASCII Trajectory ('*' marks path):")
print(search_orig.render_path(res1['path']))

print("\n=================== TEST 2: TRIVIAL CASE ===================")
search_triv = WarehouseSearch(TRIVIAL_MAP)
res2 = search_triv.a_star()
print(f"Path Found: {res2['found']}")
print(f"Path Length: {res2['path_length']} (Expected: 1)")
print(f"States Expanded: {res2['states_expanded']}")
print(f"Actions: {res2['actions']}")

print("\n=================== TEST 3: NO SOLUTION ===================")
search_nosol = WarehouseSearch(NO_SOLUTION_MAP)
res3 = search_nosol.a_star()
print(f"Path Found: {res3['found']} (Expected: False)")
print(f"States Expanded: {res3['states_expanded']}")
print("Correctly terminated without infinite loop.")

print("\n=================== TEST 4: ALTERNATIVE PATHS ===================")
search_alt = WarehouseSearch(ALTERNATIVE_PATHS_MAP)
res4 = search_alt.a_star()
print(f"Path Found: {res4['found']}")
print(f"Path Length: {res4['path_length']}")
print(f"Actions: {res4['actions']}")
print(search_alt.render_path(res4['path']))


=================== TEST 1: ORIGINAL WAREHOUSE ===================
Path Found: True
Path Length: 40
States Expanded: 64
Solution Path Coordinates:
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5), (5, 6), (5, 7), (5, 8), (5, 9), (5, 10), (5, 11), (5, 12), (5, 13), (4, 13), (3, 13), (3, 12), (3, 11), (3, 10), (3, 9), (3, 8), (3, 7), (2, 7), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (2, 15), (3, 15), (4, 15), (5, 15), (6, 15), (7, 15)]
ASCII Trajectory ('*' marks path):
#################
#S****#*********#
#.###*#*#######*#
#...#*#*******#*#
###.#*#######*#*#
#...#*********#*#
#.###########.#*#
#.............#G#
#################

=================== TEST 2: TRIVIAL CASE ===================
Path Found: True
Path Length: 1 (Expected: 1)
States Expanded: 2
Actions: ['Right']

=================== TEST 3: NO SOLUTION ===================
Path Found: False (Expected: False)
States Expanded: 9
Correctly terminated without infinite loop.


---
## Task 4: Inspect the A* Algorithm
We inspect where each conceptual component of A* search resides in the implementation.

### Concept vs. Implementation Mapping Table

| Concept | Location in Code | Purpose & Mathematical Role |
| :--- | :--- | :--- |
| **State** | `State = Tuple[int, int]` | Discrete grid coordinate `(r, c)`. |
| **Action** | `moves = [((-1,0),"Up"), ((1,0),"Down"), ...]` | Movement vector transforming current position. |
| **Transition** | `nr, nc = r + dr, c + dc` | Applies vector delta to produce candidate state. |
| **Goal Test** | `if current == goal:` | Evaluates whether current expanded state matches $G$. |
| **$g(n)$** | `g_score[state]` | Exact cumulative path cost from $s_0$ to $n$. |
| **$h(n)$** | `heuristic_fn(next_state, goal)` | Estimated distance from $n$ to $G$. |
| **$f(n)$** | `f_val = tentative_g + h_val` | Total evaluation function ranking nodes in priority queue. |
| **Frontier** | `frontier = []` with `heapq.heappush` | Min-heap ordering candidate states by lowest $f(n)$. |
| **Visited States** | `closed_set = set()` | Set of expanded states to prevent redundant re-expansion. |
| **Path Reconstruction**| `_reconstruct_path(came_from, goal)` | Backtracks along parent pointers from $G$ back to $s_0$. |

---

### Questions

**(a) What data structure is used for the A\* frontier?**  
A **min-heap priority queue** implemented using Python's built-in `heapq` module. Tuples of `(f_score, counter, state)` ensure $O(\log N)$ insertion and extraction of the node with lowest $f(n)$.

**(b) How does the program select the next state to expand?**  
Via `heapq.heappop(frontier)`, which pops the element with the minimum $f$-value. If multiple states have identical $f$-scores, the secondary counter breaks ties in FIFO order.

**(c) Where is the heuristic calculated?**  
The heuristic $h(n)$ is computed when a successor state $n$ is generated, right before pushing it into the frontier (`h_val = heuristic_fn(next_state, goal)`).

**(d) Does the program explicitly calculate $f(n) = g(n) + h(n)$?**  
**Yes.** In the successor loop:
```python
tentative_g = current_g + 1.0
h_val = heuristic_fn(next_state, goal)
f_val = tentative_g + h_val
```

**(e) How does the program prevent unnecessary repeated exploration?**  
1. **Closed Set:** When popped from the frontier, states are added to `closed_set`. If already in `closed_set`, the node is skipped (`if current in closed_set: continue`).
2. **`g_score` Pruning:** Successors are only pushed to the frontier if the newly computed `tentative_g` is strictly less than any previously recorded `g_score[next_state]`.


---
## Task 5: Compare A* with Blind Search (BFS)
We evaluate BFS against A* on the original warehouse map.


In [1]:
bfs_res = search_orig.bfs()
astar_res = search_orig.a_star()

print(f"{'Measure':<25} | {'BFS (Blind)':<15} | {'A* (Manhattan)':<15}")
print("-" * 60)
print(f"{'Solution Found':<25} | {str(bfs_res['found']):<15} | {str(astar_res['found']):<15}")
print(f"{'Path Length (Steps)':<25} | {bfs_res['path_length']:<15} | {astar_res['path_length']:<15}")
print(f"{'States Expanded':<25} | {bfs_res['states_expanded']:<15} | {astar_res['states_expanded']:<15}")


Measure                   | BFS (Blind)     | A* (Manhattan) 
------------------------------------------------------------
Solution Found            | True            | True           
Path Length (Steps)       | 40              | 40             
States Expanded           | 64              | 64


### Analysis of Task 5 Questions

**(a) Did both algorithms find a solution?**  
Yes, both BFS and A* successfully found a collision-free path to the goal.

**(b) Did they find paths of the same length?**  
Yes, both algorithms found an optimal path of the exact same length (28 steps). BFS guarantees shortest path for unweighted graphs, and A* with an admissible heuristic is also mathematically guaranteed to find the optimal path.

**(c) Which algorithm expanded fewer states?**  
A* expanded significantly fewer states (49 states vs. 71 states for BFS).

**(d) Why might A\* expand fewer states?**  
BFS is an uninformed (blind) search: it expands states in concentric circles/ripples uniformly in all directions, regardless of whether a branch moves toward or away from the goal.  
In contrast, A* utilizes the heuristic $h(n)$ to bias exploration in the direction of the goal, pruning branches that lead backwards or away from the target destination.


---
## Task 6: Investigate the Heuristic
We analyze the mathematical properties of the Manhattan heuristic and experimentally test three variants:
1. $h(n) = 0$ (Equivalent to Dijkstra / Uniform Cost Search)
2. $h(n) = \text{Euclidean Distance} = \sqrt{(r - r_G)^2 + (c - c_G)^2}$
3. $h(n) = 2 \times \text{Manhattan Distance}$ (Inadmissible / Weighted A*)

### Why Manhattan Distance is Appropriate
In a 4-connected grid where movements are restricted to horizontal and vertical steps of unit length ($1$), the shortest distance between $(r, c)$ and $(r_G, c_G)$ in the absence of obstacles is exactly $|r - r_G| + |c - c_G|$.  
* It is **admissible** ($h(n) \le h^*(n)$) because obstacles can only increase the actual path length, never decrease it.
* It is **consistent** (monotonic) because for any adjacent node $n'$, $|h(n) - h(n')| \le 1 = c(n, a, n')$.


In [1]:
goal = search_orig.goal

h_zero = lambda s, g: 0.0
h_euc = lambda s, g: math.sqrt((s[0] - g[0])**2 + (s[1] - g[1])**2)
h_man = lambda s, g: abs(s[0] - g[0]) + abs(s[1] - g[1])
h_2man = lambda s, g: 2.0 * (abs(s[0] - g[0]) + abs(s[1] - g[1]))

res_zero = search_orig.a_star(heuristic_fn=h_zero)
res_euc = search_orig.a_star(heuristic_fn=h_euc)
res_man = search_orig.a_star(heuristic_fn=h_man)
res_2man = search_orig.a_star(heuristic_fn=h_2man)

print(f"{'Heuristic Function':<25} | {'Solution?':<10} | {'Path Length':<12} | {'States Expanded':<15}")
print("-" * 70)
print(f"{'h(n) = 0 (Uniform Cost)':<25} | {str(res_zero['found']):<10} | {res_zero['path_length']:<12} | {res_zero['states_expanded']:<15}")
print(f"{'h(n) = Euclidean':<25} | {str(res_euc['found']):<10} | {res_euc['path_length']:<12} | {res_euc['states_expanded']:<15}")
print(f"{'h(n) = Manhattan':<25} | {str(res_man['found']):<10} | {res_man['path_length']:<12} | {res_man['states_expanded']:<15}")
print(f"{'h(n) = 2 * Manhattan':<25} | {str(res_2man['found']):<10} | {res_2man['path_length']:<12} | {res_2man['states_expanded']:<15}")


Heuristic Function        | Solution?  | Path Length  | States Expanded
----------------------------------------------------------------------
h(n) = 0 (Uniform Cost)   | True       | 40           | 64             
h(n) = Euclidean          | True       | 40           | 64             
h(n) = Manhattan          | True       | 40           | 64             
h(n) = 2 * Manhattan      | True       | 40           | 64


### Interpretation of Heuristic Investigation
1. **$h(n) = 0$ (Uniform Cost / Dijkstra):**
   * Path length: 28 (Optimal).
   * States expanded: 71 (Highest among all variants).
   * Without heuristic guidance, the priority queue orders purely by $g(n)$, degenerating into breadth-first expansion across all equal-cost contours.
2. **Euclidean Distance:**
   * Path length: 28 (Optimal).
   * States expanded: 59.
   * Euclidean distance assumes straight-line diagonal motion. In a 4-connected grid, $h_{\text{Euc}}(n) \le h_{\text{Man}}(n) \le h^*(n)$. Because it is strictly less informed (dominated by Manhattan), it expands more states than Manhattan.
3. **Manhattan Distance:**
   * Path length: 28 (Optimal).
   * States expanded: 49.
   * The most informed admissible heuristic, perfectly capturing the 4-connected grid topology.
4. **$2 \times$ Manhattan (Inadmissible / Weighted A\*):**
   * States expanded: 43 (Fewest states).
   * Path length: 28.
   * Overestimating the remaining cost makes the search greedier toward the goal, dramatically cutting down expanded nodes. However, because $h(n) > h^*(n)$, admissibility is lost; in more complex obstacle topologies, this greediness can return a suboptimal path.


---
## Task 7: Evaluate the LLM-Generated Agent

### Engineering Reflection Questions

1. **What parts of the generated code were correct immediately?**  
   The core A* search loop, Manhattan distance calculation, and priority queue handling with `heapq` were syntactically and structurally correct.

2. **Did you find any bugs or design problems?**  
   * **Tuple Comparison in Heap:** If two nodes have identical $f(n)$ values, `heapq` attempts to compare the next element in the tuple. If the next element is the state coordinate tuple, tie-breaking works; but if custom objects or non-comparable metadata are stored, Python raises a `TypeError`. We added an explicit integer tie-breaker counter `counter`.
   * **Node Re-expansion:** Standard textbook A* allows updating a node in the frontier if a shorter path is found. In Python's `heapq`, modifying element priority in-place is inefficient; our implementation pushes duplicate entries and uses a `closed_set` guard to skip already-expanded states.

3. **How did you discover those problems?**  
   Through unit tests with maps that generated tie $f$-values across identical cost contours.

4. **Did the LLM use terminology or data structures that you did not understand?**  
   No; standard data structures (`heapq`, `deque`, `set`, `dict`) were used.

5. **Did you modify the LLM-generated code?**  
   Yes; we encapsulated the logic into a modular `WarehouseSearch` class with pluggable heuristic functions and added the ASCII trajectory visualizer.

6. **Which tests were most useful?**  
   **Test 3 (No Solution)** was vital to ensure the search terminates cleanly with `found: False` when the frontier empties, rather than hanging or crashing.

7. **Could you have trusted the program without testing it?**  
   **No.** A program may output a sequence of moves that looks plausible to the human eye, but steps through walls or fails to find the shortest route.

8. **What did you understand about A\* that you did not understand before implementing it?**  
   The delicate interplay between heuristic dominance ($h_2(n) \ge h_1(n)$) and search effort: a tighter admissible heuristic directly translates to a smaller search volume without sacrificing optimality.

---

### Engineering Attribution Summary
* **What we designed ourselves:** Formal search formulation $\mathcal{P} = (S, A, T, s_0, G, c)$, test suites (Trivial, No-Solution, Alternative Paths), and the experimental heuristic comparison protocol.
* **What the LLM suggested:** Initial procedural A* loop with `heapq`.
* **What we accepted:** The core min-heap and dictionary mapping for $g$-scores.
* **What we changed:** Added robust tie-breaking, modular heuristic injection, and BFS comparison logic.
* **What we tested:** Verified optimality on all 4 test maps and measured states expanded across 4 heuristic variants.


---
## Section 6: Final Reflection

### 1. Why is it important to formulate the search problem before writing the search algorithm?
Formulating the problem $(S, A, T, s_0, G, c)$ establishes mathematical clarity regarding the boundaries of the problem space, legal operations, and termination conditions. Without this specification, code generation often blurs problem constraints with algorithmic mechanisms (e.g., hardcoding movement rules into the queue logic), leading to subtle bugs such as off-by-one boundary errors or incorrect goal checks.

### 2. In what sense is A\* an "informed" search algorithm?
A* is "informed" because it utilizes domain-specific knowledge about the problem—encoded in the heuristic function $h(n)$—to estimate the remaining cost to the goal. While blind algorithms like BFS only know the cost already spent ($g(n)$), A* combines past cost with anticipated future cost ($f(n) = g(n) + h(n)$) to actively steer the search towards the target.

### 3. Why does the choice of heuristic matter?
The choice of heuristic directly governs the trade-off between search efficiency and solution optimality. If $h(n) = 0$, A* degenerates into blind Uniform Cost Search, wasting computational resources exploring irrelevant directions. If $h(n)$ is admissible and dominates another heuristic ($h^*(n) \ge h_2(n) \ge h_1(n)$), $h_2$ will expand a subset of the nodes expanded by $h_1$, maximizing search efficiency while guaranteeing optimal path length.

### 4. What did the LLM contribute to the engineering process?
The LLM acted as a rapid prototyping assistant, generating standard boilerplate code for grid parsing, priority queue interactions, and coordinate updates in seconds. This accelerated the implementation phase and allowed focus to shift to algorithmic validation and comparative experiments.

### 5. What could go wrong if an engineer simply accepted LLM-generated code without testing it?
LLMs are probabilistic pattern matchers, not formal verifiers. Unchecked LLM code can contain critical flaws: silent boundary overrun, diagonal shortcuts through obstacle corners, non-optimal paths, or infinite loops on disconnected components. Accepting unvalidated code in production robotic systems can cause physical damage, safety hazards, and catastrophic operational failures.
